In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import mlflow
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score, accuracy_score

In [3]:
df = pd.read_csv("C:/Users/wwwmy/Music/Bank Fraud Detection MLOps project/Data/preprocessed_data.csv")

In [4]:
def training_and_testing(df):
    X = df.drop(columns = ['fraud_bool'])
    y = df['fraud_bool']
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)
    return X_train, X_test, y_train, y_test


In [5]:
def train_resampling(X_train,y_train):
    smt = SMOTE(random_state = 42)
    X_resampled, y_resampled = smt.fit_resample(X_train,y_train)
    return X_resampled, y_resampled

In [6]:
def model_training(X_train, y_train,ratio = 1.0,n_estimators1 = 300,max_depth1 = 6, learning_rate1 = 0.01):
    xgb_model = XGBClassifier(scale_pos_weight = ratio,random_state = 42, 
                              n_estimators = n_estimators1, max_depth = max_depth1,learning_rate = learning_rate1)
    xgb_model.fit(X_train, y_train)
    return xgb_model

In [7]:
def evaluation(xgb_model,X_test, y_test):
    y_pred = xgb_model.predict_proba(X_test)[:,1]
    pr_auc = average_precision_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test,y_pred)
    print(f"PR-AUC Score: {pr_auc}")
    print(f"roc auc: {roc_auc}")


In [8]:
# Experiment 1: SMOTE for class imbalance
X_train, X_test, y_train, y_test = training_and_testing(df)
X_train, y_train = train_resampling(X_train,y_train)
xgb_model = model_training(X_train, y_train)
evaluation(xgb_model,X_test,y_test)

PR-AUC Score: 0.06635886154710308
roc auc: 0.8283057645144003


In [9]:
# Let's check if there is overfitting or not

print("Train ROC AUC:", roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:,1]))
print("Test ROC AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

print("Train PR AUC:", average_precision_score(y_train, xgb_model.predict_proba(X_train)[:,1]))
print("Test PR AUC:", average_precision_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

Train ROC AUC: 0.9850994720635261
Test ROC AUC: 0.8283057645144003
Train PR AUC: 0.9857140265047036
Test PR AUC: 0.06635886154710308


In [10]:
# Experiment 2: Scale_pos_weight for class imbalance
X_train, X_test, y_train, y_test = training_and_testing(df)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg/pos
xgb_model = model_training(X_train, y_train, ratio)
evaluation(xgb_model,X_test,y_test)

PR-AUC Score: 0.13523233900860138
roc auc: 0.8844953965598531


In [11]:
# Let's check if there is overfitting or not

print("Train ROC AUC:", roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:,1]))
print("Test ROC AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

print("Train PR AUC:", average_precision_score(y_train, xgb_model.predict_proba(X_train)[:,1]))
print("Test PR AUC:", average_precision_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

Train ROC AUC: 0.9030283168114839
Test ROC AUC: 0.8844953965598531
Train PR AUC: 0.16643362843752524
Test PR AUC: 0.13523233900860138


In [12]:
# Scale pos weight worked better for class imbalance than SMOTE and here are the reasons:
# 1- PR AUC and ROC AUC are both higher with scale pos weight than with smote
# 2- with SMOTE, train pr auc = 0.98 and test pr auc is 0.06 which means there is severe overfitting,
# while with scale pos weight pr auc = 0.16 and roc auc = 0.13 which means there is some overfitting but it is minor

In [ ]:
for n_estimators in [100,200,300]:
    with mlflow.start_run():
        params = {'n_estimators':n_estimators,'max_depth':6,'learning_rate':0.01}
        X_train, X_test, y_train, y_test = training_and_testing(df)
        X_train, y_train = train_resampling(X_train,y_train)
        xgb_model = model_training(X_train, y_train,1,params['n_estimators'],params['max_depth'],params['learning_rate'])
        evaluation(xgb_model,X_test,y_test)
    